# Day 4: Perceptron → Multi-Layer Perceptron (MLP)

**Module 2 — Neural Network Basics | 100 Days of Data Science**

## Why This Matters
Everything from Module 1 (dot products, matrix multiplication, gradients, cross-entropy) comes together today into an actual working neural network. A perceptron is the simplest possible neural network; stacking and connecting many of them gives you an MLP — the foundation every deep learning architecture builds on.

## Topics Covered Today
1. The single perceptron
2. Perceptron limitations (why we need more than one)
3. Multi-Layer Perceptron (MLP) architecture
4. Forward pass, implemented from scratch
5. MLP in PyTorch
6. Practice exercises

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

print("NumPy version:", np.__version__)

---
## 1. The Single Perceptron

A perceptron takes inputs, multiplies each by a weight, sums them with a bias, and passes the result through an activation function.

$$z = \sum_i w_i x_i + b, \qquad \hat{y} = \text{activation}(z)$$

This is exactly the dot product + bias from Day 1, now wrapped with an activation function.

In [ ]:
def step_activation(z):
    return 1 if z >= 0 else 0

def perceptron(inputs, weights, bias):
    z = np.dot(inputs, weights) + bias
    return step_activation(z)

# Example: perceptron that mimics logical AND
weights = np.array([1, 1])
bias = -1.5

for a in [0, 1]:
    for b in [0, 1]:
        result = perceptron(np.array([a, b]), weights, bias)
        print(f"AND({a}, {b}) = {result}")

---
## 2. Perceptron Limitations

A single perceptron can only separate data with a **straight line** (it's a linear classifier). This famously fails on the XOR problem, which is not linearly separable.

In [ ]:
# Try to solve XOR with a single perceptron -- it can't be done with any single weight/bias
print("XOR truth table:")
print("0 XOR 0 = 0")
print("0 XOR 1 = 1")
print("1 XOR 0 = 1")
print("1 XOR 1 = 0")

xor_points = np.array([[0,0],[0,1],[1,0],[1,1]])
xor_labels = np.array([0,1,1,0])

plt.figure(figsize=(5,5))
for point, label in zip(xor_points, xor_labels):
    color = 'red' if label == 0 else 'blue'
    plt.scatter(*point, color=color, s=150)
plt.title('XOR: No single straight line can separate red from blue')
plt.xlim(-0.5, 1.5)
plt.ylim(-0.5, 1.5)
plt.grid(True)
plt.show()

print("\nThis is why we need MULTIPLE layers -- a single perceptron is not enough.")

---
## 3. Multi-Layer Perceptron (MLP) Architecture

An MLP stacks layers of perceptrons:
- **Input layer** — raw features
- **Hidden layer(s)** — learn intermediate representations (this is what makes non-linear problems like XOR solvable)
- **Output layer** — final prediction

Each connection between layers is a weight; each layer applies an activation function (commonly ReLU for hidden layers, softmax/sigmoid for output).

```
Input Layer        Hidden Layer        Output Layer
   x1  ----\      /--- h1 ---\
            \    /             \
             W1                 W2 ----> output
            /    \             /
   x2  ----/      \--- h2 ---/
```

---
## 4. Forward Pass From Scratch

Let's build a small MLP (2 inputs → 2 hidden neurons → 1 output) entirely with NumPy, and show it CAN solve XOR (with the right trained weights).

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def relu(z):
    return np.maximum(0, z)

class SimpleMLP:
    def __init__(self, input_size, hidden_size, output_size, seed=42):
        rng = np.random.default_rng(seed)
        self.W1 = rng.normal(size=(input_size, hidden_size))
        self.b1 = np.zeros(hidden_size)
        self.W2 = rng.normal(size=(hidden_size, output_size))
        self.b2 = np.zeros(output_size)

    def forward(self, X):
        # Layer 1: input -> hidden
        z1 = X @ self.W1 + self.b1
        a1 = relu(z1)

        # Layer 2: hidden -> output
        z2 = a1 @ self.W2 + self.b2
        a2 = sigmoid(z2)

        return a2

mlp = SimpleMLP(input_size=2, hidden_size=2, output_size=1)

X = np.array([[0,0],[0,1],[1,0],[1,1]])
output = mlp.forward(X)

print("Input:\n", X)
print("MLP output (untrained, random weights):\n", output)
print("\nNote: weights are random here, so output won't match XOR yet.")
print("After training (Day 6+), the MLP learns weights that solve this correctly.")

### Manually Set Weights That Solve XOR
To prove an MLP *can* represent XOR (it just needs training to find these weights), here's a hand-crafted solution.

In [ ]:
# Hand-crafted weights that solve XOR using 2 hidden neurons (OR and NAND combined via AND)
W1_manual = np.array([[20, -20], [20, -20]])
b1_manual = np.array([-10, 30])
W2_manual = np.array([[20], [20]])
b2_manual = np.array([-30])

def xor_mlp_forward(X):
    z1 = X @ W1_manual + b1_manual
    a1 = sigmoid(z1)
    z2 = a1 @ W2_manual + b2_manual
    a2 = sigmoid(z2)
    return a2

result = xor_mlp_forward(X)
print("XOR inputs:\n", X)
print("MLP output (rounded):\n", np.round(result).flatten())
print("Expected XOR:      \n", xor_labels)
print("\nThe MLP successfully solves XOR -- something a single perceptron never could.")

---
## 5. MLP in PyTorch

In practice, you never build this by hand — you use a framework. Here's the equivalent MLP defined with PyTorch's `nn.Module`.

In [ ]:
try:
    import torch
    import torch.nn as nn

    class MLP(nn.Module):
        def __init__(self, input_size, hidden_size, output_size):
            super().__init__()
            self.fc1 = nn.Linear(input_size, hidden_size)
            self.relu = nn.ReLU()
            self.fc2 = nn.Linear(hidden_size, output_size)
            self.sigmoid = nn.Sigmoid()

        def forward(self, x):
            x = self.fc1(x)
            x = self.relu(x)
            x = self.fc2(x)
            x = self.sigmoid(x)
            return x

    model = MLP(input_size=2, hidden_size=4, output_size=1)
    print(model)

    X_torch = torch.tensor(X, dtype=torch.float32)
    output = model(X_torch)
    print("\nOutput (untrained):\n", output)

    total_params = sum(p.numel() for p in model.parameters())
    print("\nTotal trainable parameters:", total_params)
except ImportError:
    print("PyTorch not installed. Run: pip install torch")

---
## 6. Practice Exercises
Try these before Day 5:

1. Build a single perceptron (like the AND example) that mimics logical OR.
2. Try to build a perceptron for XOR by hand-tuning weights — confirm it's impossible.
3. Extend `SimpleMLP` to have 3 hidden neurons instead of 2 and print the new output shape.
4. In the PyTorch MLP, change `hidden_size` to 8 and print how the total parameter count changes.
5. In your own words: why does adding a hidden layer let a network solve non-linearly-separable problems like XOR?

In [ ]:
# Your practice code here


---
## Summary
- A **perceptron** = dot product + bias + activation = a linear classifier
- A single perceptron **cannot** solve non-linear problems like XOR
- An **MLP** stacks layers (input → hidden → output), letting the network learn non-linear decision boundaries
- The **forward pass** is just repeated matrix multiplication + activation functions
- In practice, frameworks like PyTorch (`nn.Linear`, `nn.Module`) handle this construction for you

Next up: **Day 5 — Activation Functions (ReLU, Sigmoid, Softmax, GELU)**

---
*Part of the 100 Days of Data Science series | DL-for-Data-Science repo*